In [1]:
import os 
os.chdir('../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

!nvidia-smi

Sat Aug 23 01:55:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.76.05              Driver Version: 580.76.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
|  0%   59C    P8             41W /  450W |      39MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

In [3]:
!ls /dataset/sana/

1.6B_1024px_valid4.5	  train4.5	valid4.5
1.6B_1024px_valid4.5.zip  train4.5.zip	valid4.5.zip


In [ ]:
# ===============================
# Config
# ===============================
config = EasyDict()
config.backbone      = 'SANA'
config.train_pt_dir  = '/dataset/sana/train4.5'
config.valid_pt_dir  = '/dataset/sana/valid4.5'
config.batch_size    = 10
config.CFG           = 4.5
config.val_every     = 100
config.log_dir       = "logs/sana/0822-12:SANA,6steps,CLIP,clip_loss,scheduling"
config.latent_size = (32, 16, 16)

# for Solver
config.solver = EasyDict()
config.solver.steps = 6
config.solver.skip_type = 'time_uniform_flow'
config.solver.flow_shift = 3.0
config.solver.pred_order = 1
config.solver.corr_order = 2
config.solver.use_corrector = True

# LR & Scheduler
config.base_lr       = 2e-3
config.total_steps   = 100*1000        # 전체 학습 스텝

# Loss
config.classifier = EasyDict()

config.losses = ['inception', 'PSNR', 'clip']
config.main_loss = 'clip'

os.makedirs(config.log_dir, exist_ok=True)


In [5]:
# ===============================
# Model (frozen)
# ===============================
from backbones.dit import DiT
from backbones.sana import SANA
from utils.inception import FIDInception
from utils.clip import CLIPEmbedder

if config.backbone == 'DiT':
    model = DiT(trainable=True)
elif config.backbone == 'SANA':
    model = SANA(trainable=True)
model.set_freeze()
device = model.device
print(model)

inception = FIDInception().to(device)
if 'clip' in config.losses:
    clip = CLIPEmbedder(device=model.device)
print('done')

/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 5/5 [00:02<00:00,  2.33it/s]


done


In [ ]:
# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.taylor.solver.gdual_solver import GDual_Solver
from solvers.taylor.transform.logaffine_transform import LogAffineTransform
from solvers.taylor.extractor.table_extractor import Extractor

noise_schedule = model.get_noise_schedule()
extractor = Extractor(steps=config.solver.steps)
transform = LogAffineTransform(gamma_push=True, gamma_max=2, tau_offset=0, kappa_max=2, eps=1e-2)
solver = GDual_Solver(
    noise_schedule,
    steps=config.solver.steps,
    transform=transform,
    param_extractor=extractor,
    skip_type=config.solver.skip_type,
    flow_shift=config.solver.flow_shift,
    pred_order=config.solver.pred_order,
    corr_order=config.solver.corr_order,
    order1_kappa=True,
    order2_kappa=True,
    use_corrector=config.solver.use_corrector,
    time_learning=True,
    train_mode=True,
    checkpoint=True,
).to(device)

optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)
print('solver/optimizer')

from torch.optim.lr_scheduler import LambdaLR

start_lr    = config.base_lr          # 2e-3
end_lr      = 1e-4
decay_steps = 20_000
ratio       = end_lr / start_lr       # 0.05

def lr_lambda(step: int):
    # 0 → 20k: 선형으로 1.0 → 0.05, 그 이후 고정
    if step >= decay_steps:
        return ratio
    return 1.0 - (1.0 - ratio) * (step / decay_steps)

# global_step로 재개하는 경우 last_epoch=global_step-1로 맞추면 정확히 이어짐
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
# 재개 시 예: scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda, last_epoch=global_step-1)

print('solver/optimizer')

solver/optimizer


In [7]:
# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

if config.main_loss != 'clip':
    train_dataset = PtDataset(config.train_pt_dir)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    print('len(train_dataset) :', len(train_dataset))
    
valid_dataset = PtDataset(config.valid_pt_dir, n_files=100)
print('len(valid_dataset) :', len(valid_dataset))

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')


len(valid_dataset) : 100
dataloaders ready


In [8]:
# ===============================
# Utils
# ===============================

def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True)
        raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, optimizer):
    ckpt = {
        "global_step": int(global_step),
        "optim_state_dict": optimizer.state_dict(),
        "solver_state_dict": solver.state_dict(),
        "config": dict(config),
    }
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

In [9]:
from IPython.display import clear_output

@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    losses = {}
    if 'PSNR' in config.losses:
        losses['PSNR'] = []
    if 'inception' in config.losses:
        losses['inception'] = []
    if 'clip' in config.losses:
        losses['clip'] = []
    
    for i, batch in enumerate(valid_loader):
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features= batch['inception_feature'].to(device, non_blocking=True)
        
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                latent_pred = solver.sample(noises, model_fn)
                if 'PSNR' in losses:
                    psnr_loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                    losses['PSNR'].append(psnr_loss.item())

                if 'inception' in config.losses or 'clip' in config.losses:
                    sample_pred = model.decode_vae(latent_pred, raw_output=True)
        
                    if 'inception' in config.losses:
                        pred = inception(sample_pred)
                        inception_loss = F.mse_loss(pred, target_features)
                        losses['inception'].append(inception_loss.item())

                    if 'clip' in config.losses:
                        clip_loss = clip.get_clip_loss(sample_pred, conds)
                        losses['clip'].append(clip_loss.item())
                        print('valid :', i, clip_loss)
    clear_output()

    for key in losses:
        losses[key] = float(np.mean(losses[key]))
    return losses

In [ ]:
def do_train_loop(device, writer, solver, optimizer, global_step):
    solver.train()
    if config.main_loss == 'clip':
        prompts = np.load('prompts/mscoco2017.npz')['arr_0'].tolist()
        pbar = tqdm(range(100))
    else:
        pbar = tqdm(train_loader)
        
    for _, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        if global_step > 0 and global_step % config.val_every == 0:
        #if global_step % config.val_every == 0:
            valid_losses = get_valid_loss(device, solver)
            for key in valid_losses:
                writer.add_scalar(key, valid_losses[key], global_step)
            save_checkpoint(global_step, config.log_dir, solver, optimizer)

        optimizer.zero_grad(set_to_none=True)
        if config.main_loss == 'clip':
            noises = torch.randn(config.batch_size, *config.latent_size).to(device, non_blocking=True)
            indexes = np.random.randint(0, len(prompts), size=(len(noises),))
            conds = [prompts[index] for index in indexes]
        else:
            noises = batch['noise'].to(device, non_blocking=True)
            conds  = batch['cond']
            targets= batch['sample'].to(device, non_blocking=True)
            target_features = batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            latent_pred = solver.sample(noises, model_fn)
            if 'PSNR' == config.main_loss:
                loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                
            if 'inception' == config.main_loss or 'clip' == config.main_loss:
                sample_pred = model.decode_vae(latent_pred, raw_output=True)
    
                if 'inception' == config.main_loss:
                    pred = inception(sample_pred)
                    loss = F.mse_loss(pred, target_features)
                    
                if 'clip' == config.main_loss:
                    loss = clip.get_clip_loss(sample_pred, conds)
                    
        abort_if_bad("train", loss, global_step)  # ← 즉시 중단

        loss.backward()
        torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        optimizer.step()
        scheduler.step()   # ← lr 업데이트 포인트
        lr_now = optimizer.param_groups[0]["lr"]
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1
        
        clear_output()

    return global_step



In [11]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    while True:
        if global_step >= config.total_steps:
            break
        global_step = do_train_loop(device, writer, solver, optimizer, global_step)
    print('E-N-D')
    
if __name__ == "__main__":
    main()


tensorboard: logs/sana/0822-5:SANA,6steps,CLIP,clip_loss


  0%|          | 0/100 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 6.69 MiB is free. Including non-PyTorch memory, this process has 23.46 GiB memory in use. Of the allocated memory 22.09 GiB is allocated by PyTorch, and 947.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)